# Norwegian fisheries seasonality — Gold consumer (A10)

**Question:** How do landed fish quantity and species composition vary by Norwegian
region and month?

This notebook reads only the **published** Gold mart (the version that passed the
validation in `src/gold/validate.py`), and answers the question with **Spark SQL**.
Quantity is **round weight in tonnes**: the source's live-weight-equivalent kilograms
divided by 1,000.

Run it from the repo root or the `notebooks/` folder. It uses the storage of
`PIPELINE_ENV` (default `dev`); set `PIPELINE_STORAGE_ROOT` to read the VM's data.

## 1. Open the published Gold

In [1]:
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)  # config paths like ./data/dev are relative to the repo root

from common.config import load_config
from common.contracts import RunContext
from common.spark import get_spark
from gold.publish import published, published_tables

config = load_config(os.environ.get("PIPELINE_ENV", "dev"))
ctx = RunContext("notebook", config["env"], datetime.now(timezone.utc), config)
spark = get_spark(config)

current = published(ctx)  # (printing the full path would show your local folder names)
print("Published Gold from run", current["run_id"], "at", current["published_at"])

# Register each Gold table as a SQL view, so the queries below can use its name
for name, path in published_tables(ctx).items():
    spark.read.parquet(path).createOrReplaceTempView(name)

spark.sql("SHOW VIEWS").show(truncate=False)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/11 00:11:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Published Gold from run 64f406e6-df2f-4a10-a9c2-800b7bcb525d at 2026-11-10T22:14:36.356464+00:00
+---------+---------------------------+-----------+
|namespace|viewName                   |isTemporary|
+---------+---------------------------+-----------+
|         |dim_landing_region         |true       |
|         |dim_month                  |true       |
|         |dim_species                |true       |
|         |fact_monthly_species_region|true       |
+---------+---------------------------+-----------+



## 2. The consumer view

One SQL query joins the fact table with its three dimensions (the star schema) and
applies the filters. `:year`, `:country`, `:county` and `:species` are **parameters**:
Spark fills them in safely, instead of building the SQL text by hand.
Leave `COUNTY_CODE` / `SPECIES_CODE` empty to include all.

In [2]:
LANDING_YEAR = 2023
COUNTRY_CODE = "NOR"
COUNTY_CODE = ""    # Example: "54" for Troms og Finnmark
SPECIES_CODE = ""   # Example: "0201001" for cod

selected = spark.sql(
    """
    SELECT m.month_id, m.month, r.landing_county_code, r.landing_county_name,
           s.species_code, s.species_name,
           f.line_count, f.total_round_weight_kg,
           f.missing_round_weight_rows, f.zero_round_weight_rows
    FROM fact_monthly_species_region AS f
    JOIN dim_month AS m          USING (month_id)
    JOIN dim_landing_region AS r USING (region_id)
    JOIN dim_species AS s        USING (species_id)
    WHERE m.year = :year
      AND r.landing_country_code = :country
      AND (:county = '' OR r.landing_county_code = :county)
      AND (:species = '' OR s.species_code = :species)
    """,
    args={
        "year": LANDING_YEAR,
        "country": COUNTRY_CODE,
        "county": COUNTY_CODE,
        "species": SPECIES_CODE,
    },
)
selected.createOrReplaceTempView("selected")

26/11/11 00:11:12 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## 3. Coverage and headline result

In [3]:
spark.sql("""
    SELECT ROUND(SUM(total_round_weight_kg) / 1000, 3) AS round_weight_tonnes,
           SUM(line_count)                             AS source_lines,
           SUM(missing_round_weight_rows)              AS missing_round_weight,
           SUM(zero_round_weight_rows)                 AS zero_round_weight
    FROM selected
""").show(truncate=False)

+-------------------+------------+--------------------+-----------------+
|round_weight_tonnes|source_lines|missing_round_weight|zero_round_weight|
+-------------------+------------+--------------------+-----------------+
|2492144.338        |1063868     |1                   |168031           |
+-------------------+------------+--------------------+-----------------+



## 4. Seasonal pattern

In [4]:
spark.sql("""
    SELECT month_id,
           ROUND(SUM(total_round_weight_kg) / 1000, 3) AS round_weight_tonnes,
           SUM(line_count)                             AS source_lines
    FROM selected
    GROUP BY month_id
    ORDER BY month_id
""").show(12, truncate=False)

+--------+-------------------+------------+
|month_id|round_weight_tonnes|source_lines|
+--------+-------------------+------------+
|2023-01 |232513.779         |76007       |
|2023-02 |226964.179         |98345       |
|2023-03 |377637.044         |212536      |
|2023-04 |231849.480         |133718      |
|2023-05 |148521.423         |68235       |
|2023-06 |183910.202         |65498       |
|2023-07 |108794.002         |65543       |
|2023-08 |203165.021         |83997       |
|2023-09 |185572.568         |65272       |
|2023-10 |282289.153         |71404       |
|2023-11 |241040.534         |78821       |
|2023-12 |69886.952          |44492       |
+--------+-------------------+------------+



## 5. Largest species by round weight

In [5]:
spark.sql("""
    SELECT species_code, species_name,
           ROUND(SUM(total_round_weight_kg) / 1000, 3) AS round_weight_tonnes,
           SUM(line_count)                             AS source_lines
    FROM selected
    GROUP BY species_code, species_name
    ORDER BY round_weight_tonnes DESC
    LIMIT 10
""").show(truncate=False)

+------------+---------------------+-------------------+------------+
|species_code|species_name         |round_weight_tonnes|source_lines|
+------------+---------------------+-------------------+------------+
|0102001     |Norsk vårgytende sild|397427.791         |7201        |
|0105001     |Kolmule              |383698.045         |642         |
|0201001     |Torsk                |344357.115         |310051      |
|0104001     |Makrell              |342080.095         |6865        |
|0203001     |Sei                  |234935.876         |117905      |
|0901001     |Stortare             |173103.527         |1853        |
|0103004     |Nordsjøsild          |142097.999         |1941        |
|0202001     |Hyse                 |102976.355         |85605       |
|0505001     |Dypvannsreke         |52345.480          |35235       |
|0302003     |Snabeluer            |44218.371          |2276        |
+------------+---------------------+-------------------+------------+



## 6. Largest landing counties by round weight

Unknown county stays visible as its own row instead of being dropped.

In [6]:
spark.sql("""
    SELECT landing_county_code, landing_county_name,
           ROUND(SUM(total_round_weight_kg) / 1000, 3) AS round_weight_tonnes,
           SUM(line_count)                             AS source_lines
    FROM selected
    GROUP BY landing_county_code, landing_county_name
    ORDER BY round_weight_tonnes DESC
    LIMIT 15
""").show(truncate=False)

+-------------------+----------------------+-------------------+------------+
|landing_county_code|landing_county_name   |round_weight_tonnes|source_lines|
+-------------------+----------------------+-------------------+------------+
|54                 |Troms og Finnmark     |607963.588         |427794      |
|11                 |Rogaland              |573913.652         |39064       |
|15                 |Møre og Romsdal       |507485.081         |82568       |
|18                 |Nordland              |338684.636         |306307      |
|46                 |Vestland              |283055.291         |47745       |
|50                 |Trøndelag             |145168.936         |56665       |
|UNKNOWN            |Unknown / not supplied|30064.795          |3106        |
|42                 |Agder                 |4008.555           |47183       |
|38                 |Vestfold og Telemark  |893.581            |41354       |
|30                 |Viken                 |889.958            |

## 7. Species composition: top 3 species in each month

A **window function** ranks the species inside each month by round weight, so we can
see how the species mix changes through the year.

In [7]:
spark.sql("""
    WITH monthly AS (
        SELECT month_id, species_name, SUM(total_round_weight_kg) AS round_weight_kg
        FROM selected
        GROUP BY month_id, species_name
    ),
    ranked AS (
        SELECT month_id, species_name, round_weight_kg,
               round_weight_kg / SUM(round_weight_kg) OVER (PARTITION BY month_id) AS share,
               RANK() OVER (PARTITION BY month_id ORDER BY round_weight_kg DESC) AS rank
        FROM monthly
    )
    SELECT month_id, rank, species_name,
           ROUND(round_weight_kg / 1000, 3) AS round_weight_tonnes,
           ROUND(share * 100, 1)            AS percent_of_month
    FROM ranked
    WHERE rank <= 3
    ORDER BY month_id, rank
""").show(36, truncate=False)

+--------+----+---------------------+-------------------+----------------+
|month_id|rank|species_name         |round_weight_tonnes|percent_of_month|
+--------+----+---------------------+-------------------+----------------+
|2023-01 |1   |Norsk vårgytende sild|102175.363         |43.9            |
|2023-01 |2   |Makrell              |46526.367          |20.0            |
|2023-01 |3   |Sei                  |20329.028          |8.7             |
|2023-02 |1   |Norsk vårgytende sild|58235.788          |25.7            |
|2023-02 |2   |Torsk                |48750.456          |21.5            |
|2023-02 |3   |Kolmule              |40848.406          |18.0            |
|2023-03 |1   |Kolmule              |160339.226         |42.5            |
|2023-03 |2   |Torsk                |96640.442          |25.6            |
|2023-03 |3   |Barentshavslodde     |37631.076          |10.0            |
|2023-04 |1   |Kolmule              |93000.709          |40.1            |
|2023-04 |2   |Torsk     

## 8. Answer and limitations

*Write the answer from the results above* (which months peak, which species dominate
and how the mix changes through the year, which counties land the most).

Interpretation limits:

- A source row is a document product line, not one delivery or one fish.
- Round weight is the selected live-weight-equivalent measure; it must not be added to
  product or gross weight.
- Missing county and round weight stay explicit; zero round weight is kept and flagged
  by the Silver quality rules.

In [8]:
spark.stop()